# 04 - XGBoost para Pronóstico de Series de Tiempo

**Módulo 3 - Series de Tiempo | ML Avanzado**

Los ensambles de árboles como XGBoost no son "modelos de series de tiempo",
pero están entre los mejores pronosticadores en la práctica *una vez que el
problema se reformula como regresión supervisada* (notebook 03). Dos detalles
importan:

- Los árboles **no extrapolan**: una hoja devuelve una constante, así que no
  pueden seguir una tendencia fuera del rango visto. Nuestra serie no tiene
  tendencia de largo plazo, y además los rezagos/índice de tiempo transportan
  el nivel.
- La validación debe ser **consciente del tiempo** (`TimeSeriesSplit`), y el
  pronóstico multi-paso necesita una **estrategia** (recursiva vs directa).

In [ ]:
import os, sys, warnings
warnings.filterwarnings("ignore")

# Hacemos importable utils/ tanto si el notebook corre desde notebooks/ como
# desde la raíz del repositorio.
_here = os.getcwd()
for cand in (os.path.join(_here, "..", "utils"), os.path.join(_here, "utils"),
             os.path.join(_here, "..", "..", "module3-time-series", "utils")):
    cand = os.path.abspath(cand)
    if os.path.isdir(cand) and cand not in sys.path:
        sys.path.insert(0, cand)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mlflow
from mlflow_helpers import setup_mlflow, log_and_register, register_best_run

plt.rcParams["figure.figsize"] = (12, 4)
plt.rcParams["axes.grid"] = True
np.random.seed(42)
print("Versión de MLflow:", mlflow.__version__)

In [ ]:
# ---------------------------------------------------------------------------
# Carga del dataset UCI #235 (con caché local y respaldo sintético offline)
# ---------------------------------------------------------------------------
import io, zipfile, urllib.request

UCI_ZIP_URL = ("https://archive.ics.uci.edu/static/public/235/"
               "individual+household+electric+power+consumption.zip")

def _data_dir():
    for cand in ("../data", "data", "module3-time-series/data"):
        cand = os.path.abspath(cand)
        if os.path.isdir(cand):
            return cand
    cand = os.path.abspath("../data")
    os.makedirs(cand, exist_ok=True)
    return cand

DATA_DIR = _data_dir()
DAILY_CSV = os.path.join(DATA_DIR, "household_power_daily.csv")
HOURLY_CSV = os.path.join(DATA_DIR, "household_power_hourly.csv")

def load_household_power():
    """Devuelve (daily, hourly): potencia activa global media, en kW."""
    if os.path.isfile(DAILY_CSV) and os.path.isfile(HOURLY_CSV):
        daily = pd.read_csv(DAILY_CSV, index_col=0, parse_dates=True).iloc[:, 0]
        hourly = pd.read_csv(HOURLY_CSV, index_col=0, parse_dates=True).iloc[:, 0]
        return daily.asfreq("D"), hourly.asfreq("h")

    print("Descargando el dataset UCI #235 (~20 MB)...")
    raw = urllib.request.urlopen(UCI_ZIP_URL, timeout=180).read()
    with zipfile.ZipFile(io.BytesIO(raw)) as zf:
        with zf.open("household_power_consumption.txt") as fh:
            df = pd.read_csv(fh, sep=";", na_values=["?"], low_memory=False,
                             usecols=["Date", "Time", "Global_active_power"])
    ts = pd.to_datetime(df["Date"] + " " + df["Time"],
                        format="%d/%m/%Y %H:%M:%S")
    power = pd.Series(df["Global_active_power"].astype(float).to_numpy(),
                      index=ts, name="global_active_power_kw").sort_index()

    # Agregamos y rellenamos huecos por interpolación temporal (~1.25% de
    # minutos faltantes + un par de cortes de varios días).
    daily = power.resample("D").mean().interpolate(method="time")
    hourly = power.resample("h").mean().interpolate(method="time")
    daily = daily.iloc[1:-1]                       # primer/último día parciales
    hourly = hourly.loc[daily.index.min():
                        daily.index.max() + pd.Timedelta(hours=23)]
    daily.to_frame().to_csv(DAILY_CSV)
    hourly.to_frame().to_csv(HOURLY_CSV)
    return daily.asfreq("D"), hourly.asfreq("h")

try:
    daily, hourly = load_household_power()
    print(f"daily : {daily.index.min().date()} .. {daily.index.max().date()} "
          f"(n={len(daily)})")
    print(f"hourly: n={len(hourly)}")
except Exception as e:
    print("No se pudo descargar el dataset:", repr(e))
    print("Usando RESPALDO SINTÉTICO (estacionalidad semanal + anual).")
    rng = np.random.default_rng(7)
    idx = pd.date_range("2006-12-17", "2010-11-25", freq="D")
    t = np.arange(len(idx))
    daily = pd.Series(
        1.1
        + 0.35 * np.cos(2 * np.pi * (t - 20) / 365.25)   # invierno alto
        + 0.10 * (idx.dayofweek >= 5)                     # fin de semana
        + rng.normal(0, 0.12, len(idx)),
        index=idx, name="global_active_power_kw").clip(lower=0.1).asfreq("D")
    hidx = pd.date_range(idx.min(), idx.max() + pd.Timedelta(hours=23), freq="h")
    hh = hidx.hour.to_numpy()
    base = daily.reindex(pd.DatetimeIndex(hidx.date)).to_numpy()
    profile = 0.6 + 0.35 * np.sin(2 * np.pi * (hh - 14) / 24) \
              + 0.25 * ((hh >= 18) & (hh <= 22))
    hourly = pd.Series(base * profile + rng.normal(0, 0.05, len(hidx)),
                       index=hidx, name=daily.name).clip(lower=0.05).asfreq("h")

In [ ]:
# ---------------------------------------------------------------------------
# Métricas de pronóstico + gráfico estándar — se usan en TODOS los notebooks.
# ---------------------------------------------------------------------------
def forecast_metrics(y_true, y_pred):
    """MSE, RMSE, MAE, MAPE y sMAPE como dict {nombre: float}."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    err = y_true - y_pred
    mse = float(np.mean(err ** 2))
    return {
        "MSE":   mse,
        "RMSE":  float(np.sqrt(mse)),
        "MAE":   float(np.mean(np.abs(err))),
        "MAPE":  float(np.mean(np.abs(err) / np.abs(y_true)) * 100.0),
        "sMAPE": float(np.mean(2.0 * np.abs(err)
                               / (np.abs(y_true) + np.abs(y_pred))) * 100.0),
    }

def print_metrics(name, m):
    print(f"{name:<26s} MSE={m['MSE']:.4f}  RMSE={m['RMSE']:.4f}  "
          f"MAE={m['MAE']:.4f}  MAPE={m['MAPE']:.2f}%  sMAPE={m['sMAPE']:.2f}%")

def metrics_table(metrics_by_model):
    """dict {modelo: dict_de_métricas} -> DataFrame ordenado por sMAPE."""
    return (pd.DataFrame(metrics_by_model).T
            .sort_values("sMAPE").round(4))

def plot_forecast(train, test, forecasts, title="", tail=180, ci=None):
    """Cola del train + test real + uno o varios pronósticos.

    forecasts : dict {nombre: pd.Series indexada como test}
    ci        : tupla opcional (lower, upper) para una banda de confianza
    Devuelve la figura (útil para loggearla en MLflow).
    """
    fig, ax = plt.subplots(figsize=(13, 5))
    train.iloc[-tail:].plot(ax=ax, label="train (cola)", color="0.65")
    test.plot(ax=ax, label="real (test)", color="black", lw=2)
    for name, fc in forecasts.items():
        fc.plot(ax=ax, label=name, lw=1.8)
    if ci is not None:
        ax.fill_between(test.index, ci[0], ci[1], alpha=0.2, label="IC 95%")
    ax.set_title(title)
    ax.set_ylabel("potencia activa media (kW)")
    ax.legend()
    plt.tight_layout()
    plt.show()
    return fig

In [ ]:
# ---------------------------------------------------------------------------
# Matriz de variables supervisada para pronóstico diario (sin fuga temporal).
# Construida y explicada en el notebook 03 — aquí la reutilizamos tal cual.
# ---------------------------------------------------------------------------
def make_features(s, lags=(1, 2, 3, 7, 14, 28, 365),
                  roll_windows=(7, 28), fourier=((7, 2), (365.25, 3))):
    df = pd.DataFrame({"y": s})
    df["t_index"] = np.arange(len(df))

    # rezagos
    for L in lags:
        df[f"lag_{L}"] = df["y"].shift(L)

    # estadísticos móviles SOLO del pasado: shift(1) antes de rolling
    past = df["y"].shift(1)
    for w in roll_windows:
        df[f"rollmean_{w}"] = past.rolling(w).mean()
        df[f"rollstd_{w}"] = past.rolling(w).std()
        df[f"rollmin_{w}"] = past.rolling(w).min()
        df[f"rollmax_{w}"] = past.rolling(w).max()

    # calendario
    df["dayofweek"] = df.index.dayofweek
    df["month"] = df.index.month
    df["is_weekend"] = (df.index.dayofweek >= 5).astype(int)

    # estacionalidad de Fourier (semanal y anual)
    tt = df["t_index"].to_numpy()
    for period, K in fourier:
        for k in range(1, K + 1):
            df[f"sin_{int(period)}_{k}"] = np.sin(2 * np.pi * k * tt / period)
            df[f"cos_{int(period)}_{k}"] = np.cos(2 * np.pi * k * tt / period)
    return df

In [ ]:
feat = make_features(daily).dropna()
X, y = feat.drop(columns=["y"]), feat["y"]

H = 60
train_s, test_s = daily.iloc[:-H], daily.iloc[-H:]
print("matriz:", X.shape, "| test:", test_s.index.min().date(),
      "..", test_s.index.max().date())

## 1. Validación cruzada consciente del tiempo

In [ ]:
from sklearn.model_selection import TimeSeriesSplit
from xgboost import XGBRegressor

XGB_PARAMS = dict(n_estimators=500, learning_rate=0.05, max_depth=5,
                  subsample=0.8, colsample_bytree=0.8,
                  random_state=42, n_jobs=-1)

# CV solo sobre la parte de entrenamiento (el test de 60 días queda intacto)
mask_train = feat.index <= train_s.index.max()
X_train, y_train = X[mask_train], y[mask_train]

tscv = TimeSeriesSplit(n_splits=5, test_size=60)
cv_scores = []
for tr, va in tscv.split(X_train):
    m = XGBRegressor(**XGB_PARAMS)
    m.fit(X_train.iloc[tr], y_train.iloc[tr])
    pred = m.predict(X_train.iloc[va])
    cv_scores.append(forecast_metrics(y_train.iloc[va], pred))

cv_df = pd.DataFrame(cv_scores).round(4)
cv_df.index.name = "fold"
display(cv_df)
print("RMSE promedio CV:", round(cv_df["RMSE"].mean(), 4))

## 2. Multi-paso: recursivo vs directo

Para un horizonte $H$:

**Recursivo** — *un* modelo de 1 paso; sus predicciones se reinyectan como
rezagos para avanzar: $\hat y_{t+1} \to$ rezago $\to \hat y_{t+2} \to \dots$
*Pro*: un solo modelo, usa todos los datos. *Contra*: los errores **se
acumulan**.

**Directo** — un modelo *por horizonte* $h$, cada uno predice $y_{t+h}$ desde
variables conocidas en $t$. *Pro*: sin acumulación. *Contra*: $H$ modelos y
pronósticos que pueden quedar irregulares entre horizontes.

Implementamos la **recursiva** (la más común) para comparar de igual a igual
con SARIMA/Holt-Winters, que también pronostican 60 días "a ciegas".

In [ ]:
final = XGBRegressor(**XGB_PARAMS).fit(X_train, y_train)

In [ ]:
def xgb_recursive_forecast(model, history, h):
    """Pronóstico multi-paso RECURSIVO: extiende la serie un día a la vez,
    recalculando las variables (los rezagos recientes van siendo predichos)."""
    hist = history.copy()
    preds = []
    for _ in range(h):
        next_date = hist.index[-1] + pd.Timedelta(days=1)
        f_next = make_features(
            pd.concat([hist, pd.Series([np.nan], index=[next_date])]))
        row = f_next.drop(columns=["y"]).iloc[[-1]]
        yhat = float(model.predict(row)[0])
        hist = pd.concat([hist, pd.Series([yhat], index=[next_date])])
        preds.append(yhat)
    idx = pd.date_range(history.index[-1] + pd.Timedelta(days=1),
                        periods=h, freq="D")
    return pd.Series(preds, index=idx, name="xgb_recursive")

In [ ]:
fc_rec = xgb_recursive_forecast(final, train_s, H)
fc_rec.index = test_s.index

xgb_metrics = forecast_metrics(test_s, fc_rec)
print_metrics("XGBoost recursivo", xgb_metrics)

fig_xgb = plot_forecast(train_s, test_s, {"XGBoost recursivo": fc_rec},
                        title="XGBoost - pronóstico multi-paso recursivo (60 días)")

### Referencia: evaluación a 1 paso

La brecha entre la métrica a 1 paso (rezagos reales) y la multi-paso
(rezagos predichos) mide cuánto **se acumulan** los errores recursivos.

In [ ]:
mask_test = feat.index > train_s.index.max()
pred_1step = pd.Series(final.predict(X[mask_test]), index=y[mask_test].index)
one_step_metrics = forecast_metrics(y[mask_test], pred_1step)

comp = metrics_table({"XGB 1 paso": one_step_metrics,
                      "XGB recursivo (60 pasos)": xgb_metrics})
display(comp)

## 3. Importancia de variables

`importance_type="gain"`: mejora media del objetivo cuando la variable se usa
en un corte. (Para atribución con signo y por-predicción: valores SHAP.)

In [ ]:
imp = pd.Series(
    final.get_booster().get_score(importance_type="gain")).sort_values()
ax = imp.tail(15).plot(kind="barh", figsize=(8, 6))
ax.set_title("XGBoost - importancia por ganancia (top 15)")
plt.tight_layout(); plt.show()
# Es de esperar que dominen lag_1/lag_7, las medias móviles y Fourier anual.

## 4. Tracking + Model Registry en MLflow

Registramos el run con CV + test + figura + el modelo (flavor `xgboost`), y lo
publicamos en el registry como `module3-power-xgboost`.

In [ ]:
setup_mlflow("module3-04-xgboost", backend="dagshub")

log_and_register(
    run_name="xgboost-recursive-h60",
    params={**XGB_PARAMS, "strategy": "recursive", "horizon_days": H,
            "n_features": X.shape[1], "dataset": "uci-household-power"},
    metrics={**xgb_metrics,
             "cv_rmse_mean": float(cv_df["RMSE"].mean()),
             "one_step_RMSE": one_step_metrics["RMSE"]},
    model=final,
    flavor="xgboost",
    registered_model_name="module3-power-xgboost",
    input_example=X_train.head(3),
    tags={"notebook": "04_xgboost", "familia": "ml"},
    figures={"plots/forecast.png": fig_xgb},
)

# Alternativa: promover después el mejor run del experimento
# register_best_run("module3-04-xgboost", metric="sMAPE",
#                   registered_model_name="module3-power-xgboost", mode="min")

## 5. Serving: consumir el modelo desde el Registry

Cerramos el **ciclo de gestión del modelo**: entrenar → trackear → registrar
→ **servir**. Cargamos la última versión registrada con el wrapper genérico
**`pyfunc`** — el contrato universal de serving de MLflow: no importa el
flavor con que se guardó (xgboost, sklearn, pytorch...), el consumidor
siempre ve lo mismo, `.predict(DataFrame) -> array`. Es exactamente lo que
expone `mlflow models serve` detrás de un endpoint REST.

Como el contrato es el mismo, nuestro `xgb_recursive_forecast` funciona
**sin cambios** con el modelo servido: le da igual recibir el `XGBRegressor`
en memoria o el `pyfunc` que vino del registry.

In [ ]:
MODEL_NAME = "module3-power-xgboost"
MODEL_URI = f"models:/{MODEL_NAME}/latest"

serving_model = mlflow.pyfunc.load_model(MODEL_URI)
print("Firma del modelo (validada en cada predict):")
print(serving_model.metadata.signature)

# 1) Reproducimos el pronóstico del test con el modelo SERVIDO:
fc_serving = xgb_recursive_forecast(serving_model, train_s, H)
fc_serving.index = test_s.index
print_metrics("XGBoost servido (registry)", forecast_metrics(test_s, fc_serving))
print("¿Idéntico al modelo en memoria?",
      bool(np.allclose(fc_serving.to_numpy(), fc_rec.to_numpy())))

In [ ]:
# 2) Pronóstico REAL a futuro: en producción alimentamos TODA la serie
#    disponible y proyectamos más allá del último dato observado.
H_FUT = 30
fc_future = xgb_recursive_forecast(serving_model, daily, H_FUT)

fig, ax = plt.subplots(figsize=(12, 4))
daily.iloc[-120:].plot(ax=ax, label="observado (últimos 120 días)")
fc_future.plot(ax=ax, color="C3", lw=2, label=f"pronóstico a {H_FUT} días")
ax.axvline(daily.index[-1], color="0.5", ls="--", lw=1)
ax.set_ylabel("kW"); ax.legend()
ax.set_title(f"Serving: {MODEL_NAME}/latest pronosticando el futuro real")
plt.tight_layout(); plt.show()

# En producción este bloque ES el job batch de pronóstico: carga por nombre
# desde el registry + datos frescos -> pronóstico. Nada del entrenamiento
# viaja al consumidor; solo el nombre del modelo y el feature pipeline
# (make_features), que por eso debe versionarse junto al modelo.

## Resumen

- XGBoost pronostica vía la matriz de variables del notebook 03; validación
  con **`TimeSeriesSplit`**, nunca aleatoria.
- **Recursivo** = 1 modelo, errores que se acumulan (medimos la brecha 1 paso
  vs 60 pasos); **directo** = $H$ modelos sin acumulación.
- La **importancia por ganancia** confirma qué variables aportan (rezagos
  recientes, medias móviles, Fourier anual).
- Modelo versionado en el **Model Registry** (`module3-power-xgboost`) y
  **consumido de vuelta** vía `pyfunc` para pronosticar el futuro real —
  ciclo completo: entrenar → registrar → servir.

Siguiente: **05 — Ensambles de pronósticos**: combinar SARIMA, Holt-Winters y
XGBoost suele ganarle a cada uno por separado.